[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C50_HuggingFace_Ecosystem_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与「迷你复刻」热身

本课全程 **纯标准库 + numpy、CPU、不联网**。学一门讲库的课而不装那些库，靠两条腿：
① **迷你复刻**——把库的核心机制用一两百行自己写一遍（必跑、带 assert）；
② **真实 API 对照**——旁边给出等价的真实调用（可原样复制到有网环境）。

这个 notebook 做三件事：① 环境自检与「优雅回退」模式；② 用一个最小例子体会 **注册表 + 分发** 这个贯穿全课的模式；③ 立下本课的三条纪律。

## 1 · 环境自检与优雅回退

本课所有涉及真实库的单元格都用 `try/except ImportError` 包裹。
**有库就跑真的，没库就跑迷你版**——把这些 notebook 拷到有网环境，它们会自动升级。

In [ ]:
import sys, platform, math, json, hashlib, time, random
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np; print('numpy', np.__version__)

HAVE = {}
for name in ['torch', 'transformers', 'datasets', 'accelerate', 'peft', 'trl', 'openai']:
    try:
        mod = __import__(name)
        HAVE[name] = getattr(mod, '__version__', 'unknown')
    except ImportError:
        HAVE[name] = None

print('\n真实库可用性:')
for k, v in HAVE.items():
    print(f'  {k:<14s} {v if v else "未安装 -> 走迷你复刻路径"}')
print('\n环境就绪 ✅  —— 本课**不依赖**上述任何库，它们只是「有则更好」')

### 优雅回退的标准写法

本课每个真实 API 对照都长这样。**注意它不是 try 一个大 block，而是先探测、再分支**——
这样错误信息清晰，且不会把真正的 bug 藏进 except 里。

In [ ]:
def with_fallback(real_fn, mini_fn, lib_name, *args, **kwargs):
    '''有库就跑真的，没库就跑迷你版。返回 (结果, 走的哪条路)。'''
    if HAVE.get(lib_name):
        try:
            return real_fn(*args, **kwargs), 'real'
        except Exception as e:                 # 库在但调用失败：**要报出来**，不要静默回退
            print(f'⚠️  {lib_name} 可用但调用失败: {type(e).__name__}: {e}')
            raise
    return mini_fn(*args, **kwargs), 'mini'

def real_tokenize(text):
    from transformers import AutoTokenizer          # 只有真装了才会执行到这
    return AutoTokenizer.from_pretrained('bert-base-uncased')(text)['input_ids']

def mini_tokenize(text):
    return [hash(w) % 30000 for w in text.lower().split()]

ids, path = with_fallback(real_tokenize, mini_tokenize, 'transformers', 'Hello world')
print(f'分词结果（走 {path} 路径）: {ids}')
assert isinstance(ids, list) and len(ids) >= 2
print('\n✅ 优雅回退模式：**探测在前、分支在后**，绝不用 except 吞掉真正的 bug')

## 2 · 贯穿全课的模式：注册表 + 分发

`AutoModel` / `AutoTokenizer` / `AutoConfig` 看起来很神奇——你给一个名字，它就知道该实例化哪个类。
其实机制极简：**一张 `model_type -> 类` 的注册表，加一次字典查找**。

先把它写出来。理解了这个，模块 01 就没有黑箱了。

In [ ]:
# ── 迷你复刻：Auto* 的注册表与分发 ──
MODEL_REGISTRY = {}          # model_type -> 具体类
CONFIG_REGISTRY = {}

def register(model_type):
    '''装饰器：把一个类注册到表里。真实库用的是 OrderedDict + lazy import。'''
    def deco(cls):
        MODEL_REGISTRY[model_type] = cls
        return cls
    return deco

class MiniConfig:
    def __init__(self, model_type, **kw):
        self.model_type = model_type
        for k, v in kw.items(): setattr(self, k, v)
    def __repr__(self):
        return f'MiniConfig({self.__dict__})'

@register('bert')
class MiniBertModel:
    def __init__(self, config): self.config = config
    def kind(self): return 'encoder-only'

@register('gpt2')
class MiniGPT2Model:
    def __init__(self, config): self.config = config
    def kind(self): return 'decoder-only'

@register('t5')
class MiniT5Model:
    def __init__(self, config): self.config = config
    def kind(self): return 'encoder-decoder'

class MiniAutoModel:
    @staticmethod
    def from_config(config):
        mt = config.model_type
        if mt not in MODEL_REGISTRY:
            raise ValueError(f'Unrecognized model_type {mt!r}. '
                             f'Registered: {sorted(MODEL_REGISTRY)}')
        return MODEL_REGISTRY[mt](config)

for mt in ['bert', 'gpt2', 't5']:
    m = MiniAutoModel.from_config(MiniConfig(mt, hidden_size=768))
    print(f'{mt:<6s} -> {type(m).__name__:<16s} ({m.kind()})')

assert isinstance(MiniAutoModel.from_config(MiniConfig('bert')), MiniBertModel)
assert isinstance(MiniAutoModel.from_config(MiniConfig('t5')), MiniT5Model)
try:
    MiniAutoModel.from_config(MiniConfig('mystery-net'))
    raise RuntimeError('不该到这')
except ValueError as e:
    print(f'\n未知类型的报错: {e}')
print('\n✅ Auto* 不神奇：就是 `config.model_type` 查一次字典。')
print('   真实库的 config.json 里就有 "model_type" 字段 —— 它才是分发的依据。')

**真实 API 对照**（不在本环境运行，可原样复制到有网环境）：

```python
from transformers import AutoConfig, AutoModel, AutoTokenizer

cfg = AutoConfig.from_pretrained("bert-base-uncased")
print(cfg.model_type)          # 'bert'  ← 这就是分发依据
model = AutoModel.from_config(cfg)              # 只建结构，不加载权重
model = AutoModel.from_pretrained("bert-base-uncased")   # 建结构 + 加载权重

# 注册自定义模型（写自己的模型时用）
AutoConfig.register("my-net", MyConfig)
AutoModel.register(MyConfig, MyModel)
```

**坑**：`from_config` 只建结构不加载权重（随机初始化）；`from_pretrained` 才加载。
把两者搞混会得到一个「能跑但输出全是噪声」的模型——这个错误极其常见且难以察觉。

## 3 · 三条纪律

本课每个迷你复刻都要满足：

1. **语义对齐** —— 实现要与真实库的**契约**一致（同样的输入给同样的输出形状与边界行为），用 assert 钉死。
2. **参数账** —— 把容易搞混的参数组合算成具体数字（有效 batch、LoRA 参数量、上下文预算、限流额度）。
3. **坑要显式化** —— 每个模块都列出该层最高频的错误，并用一个可运行的反例演示它。

把第二条封装成一个小工具，后面每个模块都会用。

In [ ]:
def effective_batch(per_device, grad_accum, n_devices=1):
    '''有效 batch = 三个数的乘积。这是最常被搞混的参数组合，没有之一。'''
    return per_device * grad_accum * n_devices

print(f"{'per_device':>11s} {'accum':>6s} {'devices':>8s} {'有效batch':>9s}")
for pd, ga, nd in [(8, 1, 1), (8, 4, 1), (2, 16, 1), (8, 4, 8), (1, 32, 4)]:
    print(f'{pd:>11d} {ga:>6d} {nd:>8d} {effective_batch(pd, ga, nd):>9d}')

assert effective_batch(8, 4, 1) == effective_batch(2, 16, 1) == 32
assert effective_batch(8, 4, 8) == 256
print('\n✅ per_device=8,accum=4 与 per_device=2,accum=16 的**有效 batch 相同**（都是 32）。')
print('   前者更快（GPU 利用率高），后者更省显存。这就是显存不够时的标准换法。')
print('   ⚠️  但注意：BatchNorm 类的算子对 per_device 敏感，换了不等价（Transformer 用 LayerNorm，无此问题）。')

### 坑要显式化：一个可运行的反例

本课每个模块都会这样演示坑。先来一个：**`from_config` 与 `from_pretrained` 搞混**。

In [ ]:
class MiniWeights:
    def __init__(self, vals): self.vals = np.array(vals, dtype=float)

def mini_from_config(config):
    '''只建结构：权重随机初始化。'''
    rng = np.random.default_rng(0)
    return MiniWeights(rng.normal(size=8) * 0.02)

PRETRAINED_STORE = {'bert-base': np.arange(8, dtype=float) + 100}   # 模拟 Hub 上的权重

def mini_from_pretrained(name):
    '''建结构 + **加载权重**。'''
    m = mini_from_config(None)
    m.vals = PRETRAINED_STORE[name].copy()
    return m

a = mini_from_config(None)
b = mini_from_pretrained('bert-base')
print('from_config   权重前三个:', a.vals[:3].round(4), ' <- 随机初始化')
print('from_pretrained 权重前三个:', b.vals[:3].round(4), ' <- 真实预训练权重')
assert not np.allclose(a.vals, b.vals)
assert np.abs(a.vals).max() < 1.0, 'from_config 是小随机数'
assert b.vals[0] == 100.0, 'from_pretrained 加载了真实权重'
print('\n⚠️  两者都能「跑通」，都不会报错 —— 但 from_config 得到的是个**随机模型**。')
print('   症状：训练能跑、loss 会降，但效果远低于预期，且没有任何报错。')
print('✅ 记住：`from_config` = 只建结构；`from_pretrained` = 结构 + 权重。')

## 4 · ✏️ 练习：实现 Auto* 的「按名字前缀推断」

真实库在 `config.json` 缺失时，会退而用**模型名字**猜类型（如 `bert-base-uncased` → `bert`）。
实现 `infer_model_type(name, registry)`：把名字小写后，返回 registry 里**最长的**匹配前缀所对应的 key；
无匹配返回 `None`。

（用「最长匹配」是为了让 `gpt2-medium` 匹配 `gpt2` 而不是 `gpt`。）

In [ ]:
def infer_model_type(name, registry):
    # TODO: 小写化；在 registry 的 key 里找所有「是 name 前缀」的，返回最长的那个；无则 None
    raise NotImplementedError

In [ ]:
# —— 练习自测 ——
REG = {'bert': 1, 'gpt': 2, 'gpt2': 3, 't5': 4, 'deberta': 5, 'deberta-v2': 6}
assert infer_model_type('bert-base-uncased', REG) == 'bert'
assert infer_model_type('gpt2-medium', REG) == 'gpt2', '最长匹配：应是 gpt2 而不是 gpt'
assert infer_model_type('GPT2-XL', REG) == 'gpt2', '应大小写不敏感'
assert infer_model_type('deberta-v2-xlarge', REG) == 'deberta-v2', '最长匹配'
assert infer_model_type('deberta-base', REG) == 'deberta'
assert infer_model_type('t5-small', REG) == 't5'
assert infer_model_type('mystery-net', REG) is None
print('✅ 练习通过：这就是「名字猜类型」的全部逻辑 —— 但**优先用 config.json 的 model_type**，')
print('   名字推断只是兜底（名字可以随便改，config 不会）。')

---
### 📖 参考答案

In [ ]:
def infer_model_type(name, registry):
    n = name.lower()
    hits = [k for k in registry if n.startswith(k)]
    return max(hits, key=len) if hits else None

## 5 · 🧪 胶囊：本课会算的几笔「参数账」

预告一下后面每个模块会算清的数字。它们都是「文档里有、但组合起来才有意义」的量。

In [ ]:
ledgers = [
    ('模块 03', '有效 batch',
     'per_device × grad_accum × n_devices', effective_batch(4, 8, 2)),
    ('模块 02', '上下文预算',
     'max_length − 特殊token − 生成预留', 512 - 3 - 128),
    ('模块 04', 'LoRA 可训练参数占比',
     '2·r·(d_in+d_out) / (d_in·d_out)  [r=8, 768×768]',
     round(2 * 8 * (768 + 768) / (768 * 768) * 100, 3)),
    ('模块 04', 'LoRA 缩放因子',
     'lora_alpha / r  [alpha=16, r=8]', 16 / 8),
    ('模块 05', '限流下的最大吞吐',
     'min(RPM, TPM / 平均token数)  [RPM=60, TPM=60k, 均500]',
     min(60, 60000 // 500)),
]
print(f"{'模块':<8s} {'量':<22s} {'公式':<48s} {'值':>10s}")
for m, name, formula, val in ledgers:
    print(f'{m:<8s} {name:<22s} {formula:<48s} {str(val):>10s}')

assert effective_batch(4, 8, 2) == 64
assert abs(2 * 8 * 1536 / (768 * 768) * 100 - 0.0417 * 100) < 1.0
print('\n✅ 这五个数字覆盖了实操中最常被算错的地方。本课会逐个把它们讲清并让你自己算一遍。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：每个机制你都会 ① 用一两百行**迷你复刻**并用 assert 钉死语义，
② 对照**真实 API** 的等价写法与参数位置，③ 算清相关的**参数账**，④ 见到一个可运行的**反例**演示坑。

**接下来五个模块**：01 transformers 核心抽象 → 02 tokenizers 与 datasets →
03 Trainer 与 TrainingArguments → 04 PEFT 与 TRL → 05 Accelerate、Hub 与推理 API。

下一站：**模块 01 · transformers 核心抽象** —— `from_pretrained` 这一行到底做了什么？